# Simulating a Diffusion LLM Decoder
**Companion to the Medium post "1,107 Tokens Per Second: The LLM That Doesn't Type" (2026-09-21)**

Autoregressive models generate left to right: token *n* waits for token *n-1*.
Diffusion LLMs (like Inception's Mercury 2.5) do the opposite: they start with a
fully-masked draft and refine **all** token slots in parallel across denoising passes.

This notebook simulates that process with nothing but NumPy — no GPU, no model weights —
so you can watch parallel unmasking work step by step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

# A "sentence" is a sequence of token slots. -1 means still masked.
VOCAB = ["the","cat","sat","on","mat","dog","ran","fast","and","a","very","happy","."]
TARGET = ["the","cat","sat","on","the","mat","and","the","dog","ran","very","fast","."]
N = len(TARGET)
print("Target sentence:", " ".join(TARGET), f"({N} token slots)")

## 1. The two generation strategies, side by side
We simulate confidence for each slot as a number in [0, 1] that rises each pass —
this stands in for a real model's per-token probabilities.

In [ ]:
# Confidence each slot would have if predicted right now (fake but instructive):
# some slots are "easy" (the, and), some are "hard" (sat, ran)
easiness = np.array([0.9, 0.8, 0.35, 0.85, 0.9, 0.8, 0.9, 0.9, 0.8, 0.3, 0.55, 0.6, 0.95])

def autoregressive_passes(easiness):
    """Left-to-right: one slot committed per pass, no take-backs."""
    passes = np.zeros(len(easiness), dtype=int)
    return passes  # pass i commits slot i -> exactly N passes, fixed order

def diffusion_unmask(easiness, unmask_per_pass=3, passes=6):
    """Parallel: top-k most confident slots unmask each pass; the rest refine."""
    masked = np.ones(len(easiness), dtype=bool)
    schedule = [masked.copy()]
    conf = np.zeros(len(easiness))
    for p in range(passes):
        # confidence grows for masked slots each pass (the "refinement")
        conf[masked] += easiness[masked] * 0.45 + 0.1
        idx = np.argsort(-conf)
        newly = [i for i in idx if masked[i]][:unmask_per_pass]
        masked[newly] = False
        schedule.append(masked.copy())
        if not masked.any():
            break
    return schedule

sched = diffusion_unmask(easiness)
print(f"Autoregressive would need exactly {N} sequential commits.")
print(f"Diffusion unmasked all {N} slots in {len(sched)-1} parallel passes.")

## 2. Watch the draft fill in

In [ ]:
fig, axes = plt.subplots(len(sched), 1, figsize=(10, 1.1*len(sched)))
for ax, masked in zip(axes, sched):
    ax.set_xlim(0, N); ax.set_ylim(0, 1); ax.axis("off")
    for i in range(N):
        color = "#1a73e8" if not masked[i] else "#e0e0e0"
        ax.add_patch(plt.Rectangle((i+0.05, 0.15), 0.9, 0.7, fc=color, ec="none"))
        ax.text(i+0.5, 0.5, TARGET[i] if not masked[i] else "[M]",
                ha="center", va="center", fontsize=8,
                color="white" if not masked[i] else "#9e9e9e", weight="bold")
    n_done = (~masked).sum()
    ax.text(-0.4, 0.5, f"pass {list(sched).index(masked) if (masked==sched[0]).all() and (masked==sched[0]).all() else ''}",
            ha="right", va="center", fontsize=8, color="#616161")
fig.suptitle("Diffusion decoding: highest-confidence slots unmask first, every pass is parallel",
             fontsize=12, weight="bold")
plt.tight_layout(); plt.show()

## 3. The key trick autoregressive models can't do: fix early mistakes
Commit to slot 0 in pass 1. In pass 3, the model realizes slot 0 was wrong.
Autoregressive: stuck forever. Diffusion: just re-mask and refine it.

In [ ]:
# Simulate: after pass 2, slot 0's "true" word changes (new evidence arrived)
draft = np.array(TARGET, dtype=object)
masked = np.zeros(N, dtype=bool)
print("Pass 2 draft:", " ".join(draft))
print("New evidence: the sentence actually starts with 'a', not 'the'.")
# Autoregressive answer: cannot change committed token 0
ar_fix = " ".join(draft)
# Diffusion answer: re-mask slot 0, refine, unmask corrected
masked[0] = True
draft[0] = "[M]"
draft[0] = "a"   # next pass predicts corrected token with full bidirectional context
masked[0] = False
print("Autoregressive (locked):", ar_fix)
print("Diffusion (re-masked):  ", " ".join(draft))

## 4. Why this is a speed story: the throughput math
Mercury 2.5's claimed 1,107 tok/s vs independent measurements of autoregressive flagships.
Each bar pair below = time to generate a fixed answer; lower is better.

In [ ]:
models = {"Mercury 2.5 (dLLM, vendor)": 1107,
          "Gemini 3.8 Flash (indep.)": 277.5,
          "DeepSeek V4.1 Flash (indep.)": 89,
          "Claude Fable 5.1 (indep.)": 65.1,
          "GPT-6 Astra med (indep.)": 50.4}
answer_tokens = 500  # a long answer / small agent trace
times = {m: answer_tokens / s for m, s in models.items()}
fig, ax = plt.subplots(figsize=(9, 4.5))
names = list(times); vals = [times[n] for n in names]
bars = ax.barh(names, vals, color=["#e8710a"] + ["#1a73e8"]*4, alpha=0.9)
ax.set_xlabel(f"seconds to generate {answer_tokens} tokens (lower = faster)")
ax.set_title(f"{answer_tokens}-token answer: generation time by model", loc="left", weight="bold")
for b, v in zip(bars, vals):
    ax.text(v + 0.1, b.get_y() + b.get_height()/2, f"{v:.1f}s", va="center", weight="bold")
ax.set_xlim(0, max(vals)*1.25)
plt.tight_layout(); plt.show()
print(f"Mercury 2.5 is ~{times['GPT-6 Astra med (indep.)']/times['Mercury 2.5 (dLLM, vendor)']:.0f}x faster than GPT-6 Astra-class speed on this answer length.")
print("Caveat: diffusion figures are vendor-reported; autoregressive figures are independent.")

## 5. Try it with a real model (optional)
If you have `transformers` installed, this cell runs a real masked-language model
(BERT) doing one manual denoising pass — the same primitive Mercury 2.5 scales up.

In [ ]:
try:
    from transformers import pipeline
    fill = pipeline("fill-mask", model="bert-base-uncased")
    sent = "The [MASK] sat on the [MASK] and watched the [MASK] run."
    print("Draft:", sent)
    for part in fill(sent):
        print(f"  {part['token_str'].strip():>10}  score={part['score']:.3f}  ->",
              sent.replace("[MASK]", part['token_str'].strip(), 1)[:60], "...")
except Exception as e:
    print("transformers not available in this environment — skipping.", type(e).__name__)
    print("Install with: pip install transformers torch  (CPU is fine for bert-base)")

## Takeaway
Diffusion decoding trades one constraint (fixed answer length, ~32 parallel passes)
for freedom from the sequential bottleneck. At 1,107 tok/s claimed throughput, that
trade is why Mercury 2.5 targets voice agents, RAG pipelines, and subagent swarms —
workloads where the model is called dozens of times per user answer.

*Numbers: Inception Labs Mercury 2.5 launch (Sept 8, 2026); Artificial Analysis and
Requesty.ai independent speed trackers. Vendor figures are self-reported.*